# ML Zoomcamp 2025 — Module 2 Homework

Dataset: [Car Fuel Efficiency](https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv)

Goal: build a regression model to predict `fuel_efficiency_mpg`.

Full instructions: [[Courses/machine-learning-zoomcamp/cohorts/2025/02-regression/homework.md]]

> Note: if your answer doesn't match an option exactly, pick the closest one. If it's exactly
> in between two options, pick the higher value.


## Setup


In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

%matplotlib inline


## Preparing the dataset

Download the data and keep only these columns:

* `engine_displacement`
* `horsepower`
* `vehicle_weight`
* `model_year`
* `fuel_efficiency_mpg`


In [2]:
url = 'https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv'
df = pd.read_csv(url)
df.head()


In [5]:
columns = [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year',
    'fuel_efficiency_mpg',
]

df = df[columns]
df.head()


In [11]:
df.isnull().sum()

df['fuel_efficiency_mpg']


0       13.231729
1       13.688217
2       14.246341
3       16.912736
4       12.488369
          ...
9699    15.101802
9700    17.962326
9701    17.186587
9702    15.331551
9703    14.884467
Name: fuel_efficiency_mpg, Length: 9704, dtype: float64


## EDA

Look at the `fuel_efficiency_mpg` variable. Does it have a long tail?


In [14]:
sns.histplot(df['fuel_efficiency_mpg'])
# No long tail, normal dist


<Axes: xlabel='fuel_efficiency_mpg', ylabel='Count'>


## Question 1. Column with missing values

There's one column with missing values. What is it?

- `engine_displacement`
- `horsepower`
- `vehicle_weight`
- `model_year`


In [ ]:
# horsepower


## Question 2. Median horsepower

What's the median (50% percentile) for variable `horsepower`?

- 49
- 99
- 149
- 199


In [15]:
df['horsepower'].median()


np.float64(149.0)


## Prepare and split the dataset

* Shuffle the dataset (the filtered one from above) using seed `42`.
* Split into train/val/test with a 60%/20%/20% distribution.
* Use the same shuffle-and-split approach as in the lectures.


In [71]:
# Shuffle with seed 42, split into 60/20/20 train/val/test
np.random.seed(42)

indexes = (np.arange(df.shape[0]))
np.random.shuffle(indexes)
indexes = indexes.tolist()

val_size = int(df.shape[0] * 0.2)

df_val = df.iloc[indexes[:val_size]]
df_test = df.iloc[indexes[val_size: val_size + val_size]]
df_train = df.iloc[indexes[val_size * 2:]]

# print(df_val.shape, df_test.shape, df_train.shape)
y_val = df_val['fuel_efficiency_mpg']
y_test = df_test['fuel_efficiency_mpg']
y_train = df_train['fuel_efficiency_mpg']

del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']
del df_train['fuel_efficiency_mpg']


## Question 3. Filling missing values: 0 vs. mean

* We need to deal with missing values for the column identified in Q1.
* Try two options: fill with `0`, or fill with the mean of the training data only.
* For each option, train a linear regression model without regularization (the code from the
  lessons) and evaluate RMSE on the validation set.
* Round each RMSE to 2 decimal places (`round(score, 2)`).
* Which option gives the better (lower) RMSE?

- With 0
- With mean
- Both are equally good


In [74]:
def prepare_X(df):
    df = df.copy()

    # fill in null values

    X = np.column_stack([np.ones(df.shape[0]), df])
    return X

def train_linear_regression(X, y):
    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)

    # print(y.shape)
    # print(X.shape)
    return XTX_inv.dot(X.T).dot(y)

X_train = prepare_X(df_train)
X_train
w = train_linear_regression(X_train, y_train)

def rmse(y1, y2):
    square_error = (y1 - y2) ** 2
    mean_square_error = square_error.mean()

    return np.sqrt(mean_square_error)


In [102]:
# Option A: fill missing values with 0, train, evaluate RMSE on validation
def prepare_X(df):
    df = df.copy()
    # print(df['horsepower'].isnull().sum())
    df['horsepower'] = df['horsepower'].fillna(0)
    # print(df['horsepower'].isnull().sum())

    X = np.column_stack([np.ones(df.shape[0]), df])
    return X

X_train = prepare_X(df_train)
w = train_linear_regression(X_train, y_train)
# y_train_pred = X_train.dot(w)

# sns.histplot(y_train_pred, color='blue')
# sns.histplot(y_train, color='green')
# 
X_val = prepare_X(df_val)
y_val_pred = X_val.dot(w)

sns.histplot(y_val_pred, color='blue')
sns.histplot(y_val, color='green')

val_rmse = rmse(y_val_pred, y_val)
val_rmse


np.float64(0.5304790083143435)


In [103]:
# Option B: fill missing values with the training mean, train, evaluate RMSE on validation
def prepare_X(df):
    df = df.copy()
    # print(df['horsepower'].isnull().sum())
    df['horsepower'] = df['horsepower'].fillna(df['horsepower'].mean())
    # print(df['horsepower'].isnull().sum())

    X = np.column_stack([np.ones(df.shape[0]), df])
    return X

X_train = prepare_X(df_train)
w = train_linear_regression(X_train, y_train)
# y_train_pred = X_train.dot(w)

# sns.histplot(y_train_pred, color='blue')
# sns.histplot(y_train, color='green')
# 
X_val = prepare_X(df_val)
y_val_pred = X_val.dot(w)

sns.histplot(y_val_pred, color='blue')
sns.histplot(y_val, color='green')

val_rmse = rmse(y_val_pred, y_val)
val_rmse


np.float64(0.470479377750519)


## Question 4. Regularized linear regression

* Fill NAs with `0` for this question.
* Try different values of `r` from `[0, 0.01, 0.1, 1, 5, 10, 100]`.
* Evaluate each on the validation set with RMSE, rounded to 2 decimal places.
* Which `r` gives the best RMSE? (If tied, pick the smallest `r`.)

- 0
- 0.01
- 1
- 10
- 100


In [116]:
# Loop over r values, train a regularized model for each, print RMSE
def prepare_X(df):
    df = df.copy()
    # print(df['horsepower'].isnull().sum())
    df['horsepower'] = df['horsepower'].fillna(0)
    # print(df['horsepower'].isnull().sum())

    X = np.column_stack([np.ones(df.shape[0]), df])
    return X

def train_linear_regression(X, y, r=0):
    XTX = X.T.dot(X)

    XTX = XTX + (r * np.eye(XTX.shape[0]))
    
    XTX_inv = np.linalg.inv(XTX)
    # print(y.shape)
    # print(X.shape)
    return XTX_inv.dot(X.T).dot(y)

lst = []
    
for r in [0, 0.01, 0.1, 1, 10, 100]:
    lst.append(round(rmse(prepare_X(df_val).dot(train_linear_regression(prepare_X(df_train), y_train, r)), y_val), 2))

print(lst)

print("Min: " + str(min(lst)))


[np.float64(0.53), np.float64(0.53), np.float64(0.53), np.float64(0.54), np.float64(0.54), np.float64(0.54)]
Min: 0.53


## Question 5. Effect of the seed on stability

* Try seed values `[0, 1, ..., 9]`.
* For each seed, redo the 60/20/20 split, fill NAs with 0, train without regularization, and
  evaluate RMSE on the validation set.
* Compute the standard deviation of the 10 RMSE scores with `np.std`, rounded to 3 decimals.

- 0.001
- 0.006
- 0.060
- 0.600

> Standard deviation shows how different the values are. Low = model is stable across splits.


In [120]:
# Loop over seeds 0-9, collect validation RMSE per seed, compute np.std
# 
result = []
for seed in range(10):
    np.random.seed(seed)
    
    indexes = (np.arange(df.shape[0]))
    np.random.shuffle(indexes)
    indexes = indexes.tolist()
    
    val_size = int(df.shape[0] * 0.2)
    
    df_val = df.iloc[indexes[:val_size]]
    df_test = df.iloc[indexes[val_size: val_size + val_size]]
    df_train = df.iloc[indexes[val_size * 2:]]
    
    # print(df_val.shape, df_test.shape, df_train.shape)
    y_val = df_val['fuel_efficiency_mpg']
    y_test = df_test['fuel_efficiency_mpg']
    y_train = df_train['fuel_efficiency_mpg']
    
    del df_val['fuel_efficiency_mpg']
    del df_test['fuel_efficiency_mpg']
    del df_train['fuel_efficiency_mpg']

    def prepare_X(df):
        df = df.copy()
        # print(df['horsepower'].isnull().sum())
        df['horsepower'] = df['horsepower'].fillna(0)
        # print(df['horsepower'].isnull().sum())
    
        X = np.column_stack([np.ones(df.shape[0]), df])
        return X
    
    def train_linear_regression(X, y, r=0):
        XTX = X.T.dot(X)
    
        XTX = XTX + (r * np.eye(XTX.shape[0]))
        
        XTX_inv = np.linalg.inv(XTX)
        # print(y.shape)
        # print(X.shape)
        return XTX_inv.dot(X.T).dot(y)
        
    result.append(round(rmse(prepare_X(df_val).dot(train_linear_regression(prepare_X(df_train), y_train, r)), y_val), 2))
    
np.array(result).std()


np.float64(0.008062257748298557)


## Question 6. Final model on the test set

* Split with seed `9`.
* Combine train and validation into one training set.
* Fill NAs with `0`, train with `r=0.001`.
* What's the RMSE on the test set?

- 0.15
- 0.515
- 5.15
- 51.5


In [127]:
# Seed 9 split, combine train+val, train with r=0.001, evaluate RMSE on test
np.random.seed(9)

indexes = (np.arange(df.shape[0]))
np.random.shuffle(indexes)
indexes = indexes.tolist()

val_size = int(df.shape[0] * 0.2)

df_val = df.iloc[indexes[:val_size]]
df_test = df.iloc[indexes[val_size: val_size + val_size]]
df_train = df.iloc[indexes[val_size * 2:]]

# print(df_val.shape, df_test.shape, df_train.shape)
y_val = df_val['fuel_efficiency_mpg']
y_test = df_test['fuel_efficiency_mpg']
y_train = df_train['fuel_efficiency_mpg']

del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']
del df_train['fuel_efficiency_mpg']

def prepare_X(df):
    df = df.copy()
    # print(df['horsepower'].isnull().sum())
    df['horsepower'] = df['horsepower'].fillna(0)
    # print(df['horsepower'].isnull().sum())

    X = np.column_stack([np.ones(df.shape[0]), df])
    return X

def train_linear_regression(X, y, r=0.001):
    XTX = X.T.dot(X)

    XTX = XTX + (r * np.eye(XTX.shape[0]))
    
    XTX_inv = np.linalg.inv(XTX)
    # print(y.shape)
    # print(X.shape)
    return XTX_inv.dot(X.T).dot(y)

df_full = pd.concat([df_train, df_val])
y_full = pd.concat([y_train, y_val])

X_full = prepare_X(df_full)
w = train_linear_regression(X_full, y_full)

X_test = prepare_X(df_test)
y_pred = X_test.dot(w)

sns.histplot(y_pred, color='blue')
sns.histplot(y_test, color='green')

test_rmse = rmse(y_pred, y_test)
test_rmse


np.float64(0.5197377167065795)


## Submit the results

* Submit here: https://courses.datatalks.club/ml-zoomcamp-2025/homework/hw02
* If your answer doesn't match an option exactly, pick the closest one. If exactly in between two
  options, pick the higher value.
